# optimizer-init-params-list — faded example 1: Materialize params into a list in __init__

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-init-params-list`. The last cell reports your progress on the `PyTorch: Optimizer init` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Optimizer init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-init-params-list`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-init-params-list"
DD_SUBTOPIC = "PyTorch: Optimizer init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Storing `self.params = list(params)` in an optimizer's `__init__` converts any iterable (including one-shot generators) into a re-usable list. Without this, a generator passed as `params` is exhausted after the first `.step()`, and all subsequent steps silently skip the update loop.

## Faded exercise 1

Complete `SimpleSGD.__init__`. Store the learning rate AND materialize the params iterable into a Python list. The `step` and `zero_grad` bodies are already provided.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn

class SimpleSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None

# --- run it ---
t.manual_seed(0)
model = nn.Linear(4, 2)
opt = SimpleSGD(model.parameters(), lr=0.1)
print(type(opt.params).__name__)  # list
print(len(opt.params))  # 2 (weight + bias)


def _test():
    import torch as t
    import torch.nn as nn
    t.manual_seed(0)
    model = nn.Linear(4, 2)
    opt = SimpleSGD(model.parameters(), lr=0.1)
    assert isinstance(opt.params, list), 'params must be a list'
    assert len(opt.params) == 2, 'Linear(4,2) has 2 param tensors (weight + bias)'
    assert opt.lr == 0.1
    # Run 3 steps and confirm params change each time
    prev = [p.data.clone() for p in opt.params]
    for _ in range(3):
        for p in opt.params:
            p.grad = t.ones_like(p)
        opt.step()
        curr = [p.data.clone() for p in opt.params]
        for old, new in zip(prev, curr):
            assert not t.allclose(old, new), 'params should update each step'
        prev = curr
    opt.zero_grad()
    for p in opt.params:
        assert p.grad is None


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class SimpleSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None

# --- run it ---
t.manual_seed(0)
model = nn.Linear(4, 2)
opt = SimpleSGD(model.parameters(), lr=0.1)
print(type(opt.params).__name__)  # list
print(len(opt.params))  # 2 (weight + bias)
```
</details>